# Telco Churn Prediction

In [5]:
# Install missing libraries
!pip install -q catboost optuna lightgbm xgboost joblib

In [ ]:
import numpy as np
import pandas as pd
import time
import joblib
import os
import json
from itertools import combinations
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold, train_test_split
from sklearn.preprocessing import TargetEncoder
from lightgbm import LGBMClassifier, early_stopping
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.linear_model import LogisticRegression
import optuna

# 1. PREPROCESSOR CLASS

class Preprocessor:
    def __init__(self, target_col='Churn'):
        self.target_col = target_col
        self.cats = [
            'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService',
            'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
            'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
            'Contract', 'PaperlessBilling', 'PaymentMethod'
        ]
        self.nums = ['tenure', 'MonthlyCharges', 'TotalCharges']
        self.top_cats_for_ngram = [
            'Contract', 'InternetService', 'PaymentMethod',
            'OnlineSecurity', 'TechSupport', 'PaperlessBilling'
        ]
        self.new_nums = []
        self.num_as_cat = [f'CAT_{col}' for col in self.nums]
        self.ngram_cols = []
        for c1, c2 in combinations(self.top_cats_for_ngram, 2):
            self.ngram_cols.append(f"BG_{c1}_{c2}")
        top4 = self.top_cats_for_ngram[:4]
        for c1, c2, c3 in combinations(top4, 3):
            self.ngram_cols.append(f"TG_{c1}_{c2}_{c3}")
        self.digit_cols = [
            'tenure_first_digit', 'tenure_last_digit', 'tenure_second_digit',
            'tenure_mod10', 'tenure_mod12', 'tenure_num_digits',
            'tenure_is_multiple_10', 'tenure_rounded_10', 'tenure_dev_from_round10',
            'mc_first_digit', 'mc_last_digit', 'mc_second_digit',
            'mc_mod10', 'mc_mod100', 'mc_num_digits', 
            'mc_is_multiple_10', 'mc_is_multiple_50',
            'mc_rounded_10', 'mc_fractional', 'mc_dev_from_round10',
            'tc_first_digit', 'tc_last_digit', 'tc_second_digit',
            'tc_mod10', 'tc_mod100', 'tc_num_digits',
            'tc_is_multiple_10', 'tc_is_multiple_100',
            'tc_rounded_100', 'tc_fractional', 'tc_dev_from_round100',
            'tenure_years', 'tenure_months_in_year', 'mc_per_digit', 'tc_per_digit'
        ]
        self.dist_features = [
            'pctrank_nonchurner_TC', 'zscore_churn_gap_TC', 'pctrank_churn_gap_TC',
            'resid_IS_MC', 'cond_pctrank_IS_TC', 'zscore_nonchurner_TC',
            'pctrank_orig_TC', 'pctrank_churner_TC', 'cond_pctrank_C_TC'
        ]
        self.q_features = []
        for q_label in ['q25', 'q50', 'q75']:
            self.q_features.extend([f'dist_To_ch_{q_label}', f'dist_To_nc_{q_label}', f'qdist_gap_To_{q_label}'])

        self.is_fitted = False
        self.freq_maps = {}
        self.churner_tc = None
        self.nonchurner_tc = None
        self.tc = None
        self.is_mc_mean = None
        self.cond_pctrank_refs = {}
        self.quantile_refs = {}
        self.total_charges_median = None

    @staticmethod
    def pctrank_against(values, reference):
        ref_sorted = np.sort(reference)
        return (np.searchsorted(ref_sorted, values) / len(ref_sorted)).astype('float32')

    @staticmethod
    def zscore_against(values, reference):
        mu, sigma = np.mean(reference), np.std(reference)
        return (np.zeros(len(values), dtype='float32') if sigma == 0 
                else ((values - mu) / sigma).astype('float32'))

    def _fit_stats(self, train):
        for col in self.nums:
            self.freq_maps[col] = train[col].value_counts(normalize=True).astype('float32')
        self.churner_tc = train.loc[train[self.target_col] == 1, 'TotalCharges'].values.astype('float32')
        self.nonchurner_tc = train.loc[train[self.target_col] == 0, 'TotalCharges'].values.astype('float32')
        self.tc = train['TotalCharges'].values.astype('float32')
        self.is_mc_mean = train.groupby('InternetService')['MonthlyCharges'].mean().astype('float32')
        self.total_charges_median = train['TotalCharges'].median()
        self.cond_pctrank_refs = {}
        for cat_col in ['InternetService', 'Contract']:
            self.cond_pctrank_refs[cat_col] = {}
            for cat_val in train[cat_col].unique():
                ref = train.loc[train[cat_col] == cat_val, 'TotalCharges'].values.astype('float32')
                self.cond_pctrank_refs[cat_col][cat_val] = ref
        self.quantile_refs = {}
        for q_label, q_val in [('q25', 0.25), ('q50', 0.50), ('q75', 0.75)]:
            self.quantile_refs[q_label] = {
                'ch': np.quantile(self.churner_tc, q_val),
                'nc': np.quantile(self.nonchurner_tc, q_val)
            }
        self.is_fitted = True

    def _create_frequency_encoding(self, df):
        for col in self.nums:
            df[f'FREQ_{col}'] = df[col].map(self.freq_maps[col]).fillna(0).astype('float32')

    def _create_arithmetic_interactions(self, df):
        df['charges_deviation'] = (df['TotalCharges'] - df['tenure'] * df['MonthlyCharges']).astype('float32')
        df['monthly_to_total_ratio'] = (df['MonthlyCharges'] / (df['TotalCharges'] + 1)).astype('float32')
        df['avg_monthly_charges'] = (df['TotalCharges'] / (df['tenure'] + 1)).astype('float32')
        df['cost_per_service'] = (df['MonthlyCharges'] / (df['service_count'] + 1)).astype('float32')
        df['total_per_service'] = (df['TotalCharges'] / (df['service_count'] + 1)).astype('float32')

    def _create_service_counts(self, df):
        service_cols = ['PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup',
                        'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
        df['service_count'] = (df[service_cols] == 'Yes').sum(axis=1).astype('float32')
        df['has_internet'] = (df['InternetService'] != 'No').astype('float32')
        df['has_phone'] = (df['PhoneService'] == 'Yes').astype('float32')

    def _apply_distribution_features(self, df):
        tc = df['TotalCharges'].values
        df['pctrank_nonchurner_TC'] = self.pctrank_against(tc, self.nonchurner_tc)
        df['pctrank_churner_TC'] = self.pctrank_against(tc, self.churner_tc)
        df['pctrank_orig_TC'] = self.pctrank_against(tc, self.tc)
        df['zscore_churn_gap_TC'] = (np.abs(self.zscore_against(tc, self.churner_tc)) - 
                                     np.abs(self.zscore_against(tc, self.nonchurner_tc))).astype('float32')
        df['zscore_nonchurner_TC'] = self.zscore_against(tc, self.nonchurner_tc)
        df['pctrank_churn_gap_TC'] = (self.pctrank_against(tc, self.churner_tc) - 
                                      self.pctrank_against(tc, self.nonchurner_tc)).astype('float32')
        df['resid_IS_MC'] = (df['MonthlyCharges'] - df['InternetService'].map(self.is_mc_mean).fillna(0)).astype('float32')
        for cat_col, name in [('InternetService', 'cond_pctrank_IS_TC'), ('Contract', 'cond_pctrank_C_TC')]:
            vals = np.zeros(len(df), dtype='float32')
            for cat_val, ref in self.cond_pctrank_refs[cat_col].items():
                mask = df[cat_col] == cat_val
                if mask.sum() > 0 and len(ref) > 0:
                    vals[mask] = self.pctrank_against(df.loc[mask, 'TotalCharges'].values, ref)
            df[name] = vals

    def _apply_quantile_distance_features(self, df):
        for q_label, q_dict in self.quantile_refs.items():
            ch_q = q_dict['ch']
            nc_q = q_dict['nc']
            df[f'dist_To_ch_{q_label}'] = np.abs(df['TotalCharges'] - ch_q).astype('float32')
            df[f'dist_To_nc_{q_label}'] = np.abs(df['TotalCharges'] - nc_q).astype('float32')
            df[f'qdist_gap_To_{q_label}'] = (df[f'dist_To_nc_{q_label}'] - df[f'dist_To_ch_{q_label}']).astype('float32')

    def _create_digit_features(self, df):
        t_str = df['tenure'].astype(str)
        df['tenure_first_digit'] = t_str.str[0].astype(int)
        df['tenure_last_digit'] = t_str.str[-1].astype(int)
        df['tenure_second_digit'] = t_str.apply(lambda x: int(x[1]) if len(x) > 1 else 0)
        df['tenure_mod10'] = df['tenure'] % 10
        df['tenure_mod12'] = df['tenure'] % 12
        df['tenure_num_digits'] = t_str.str.len()
        df['tenure_is_multiple_10'] = (df['tenure'] % 10 == 0).astype('float32')
        df['tenure_rounded_10'] = np.round(df['tenure'] / 10) * 10
        df['tenure_dev_from_round10'] = np.abs(df['tenure'] - df['tenure_rounded_10'])
        mc_str = df['MonthlyCharges'].astype(str).str.replace('.', '', regex=False)
        df['mc_first_digit'] = mc_str.str[0].astype(int)
        df['mc_last_digit'] = mc_str.str[-1].astype(int)
        df['mc_second_digit'] = mc_str.apply(lambda x: int(x[1]) if len(x) > 1 else 0)
        df['mc_mod10'] = np.floor(df['MonthlyCharges']) % 10
        df['mc_mod100'] = np.floor(df['MonthlyCharges']) % 100
        df['mc_num_digits'] = np.floor(df['MonthlyCharges']).astype(int).astype(str).str.len()
        df['mc_is_multiple_10'] = (np.floor(df['MonthlyCharges']) % 10 == 0).astype('float32')
        df['mc_is_multiple_50'] = (np.floor(df['MonthlyCharges']) % 50 == 0).astype('float32')
        df['mc_rounded_10'] = np.round(df['MonthlyCharges'] / 10) * 10
        df['mc_fractional'] = df['MonthlyCharges'] - np.floor(df['MonthlyCharges'])
        df['mc_dev_from_round10'] = np.abs(df['MonthlyCharges'] - df['mc_rounded_10'])
        tc_str = df['TotalCharges'].astype(str).str.replace('.', '', regex=False)
        df['tc_first_digit'] = tc_str.str[0].astype(int)
        df['tc_last_digit'] = tc_str.str[-1].astype(int)
        df['tc_second_digit'] = tc_str.apply(lambda x: int(x[1]) if len(x) > 1 else 0)
        df['tc_mod10'] = np.floor(df['TotalCharges']) % 10
        df['tc_mod100'] = np.floor(df['TotalCharges']) % 100
        df['tc_num_digits'] = np.floor(df['TotalCharges']).astype(int).astype(str).str.len()
        df['tc_is_multiple_10'] = (np.floor(df['TotalCharges']) % 10 == 0).astype('float32')
        df['tc_is_multiple_100'] = (np.floor(df['TotalCharges']) % 100 == 0).astype('float32')
        df['tc_rounded_100'] = np.round(df['TotalCharges'] / 100) * 100
        df['tc_fractional'] = df['TotalCharges'] - np.floor(df['TotalCharges'])
        df['tc_dev_from_round100'] = np.abs(df['TotalCharges'] - df['tc_rounded_100'])
        df['tenure_years'] = df['tenure'] // 12
        df['tenure_months_in_year'] = df['tenure'] % 12
        df['mc_per_digit'] = df['MonthlyCharges'] / (df['mc_num_digits'] + 0.001)
        df['tc_per_digit'] = df['TotalCharges'] / (df['tc_num_digits'] + 0.001)

    def _create_num_as_cat(self, df):
        for col in self.nums:
            df[f'CAT_{col}'] = df[col].astype(str).astype('category')

    def _create_ngram_features(self, df):
        for c1, c2 in combinations(self.top_cats_for_ngram, 2):
            df[f"BG_{c1}_{c2}"] = (df[c1].astype(str) + "_" + df[c2].astype(str)).astype('category')
        top4 = self.top_cats_for_ngram[:4]
        for c1, c2, c3 in combinations(top4, 3):
            df[f"TG_{c1}_{c2}_{c3}"] = (df[c1].astype(str) + "_" + df[c2].astype(str) + "_" + df[c3].astype(str)).astype('category')

    def fit_transform(self, train, test, orig=None):
        for df in [train, test] + ([orig] if orig is not None else []):
            if df is not None and self.target_col in df.columns:
                if not pd.api.types.is_numeric_dtype(df[self.target_col]):
                    df[self.target_col] = df[self.target_col].astype(str).str.strip().str.capitalize().map({'No': 0, 'Yes': 1}).fillna(0).astype(int)
            if df is not None and 'TotalCharges' in df.columns:
                df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
                df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())
        self._fit_stats(train)
        self._create_service_counts(train); self._create_service_counts(test)
        self.new_nums += ['service_count', 'has_internet', 'has_phone']
        self._create_frequency_encoding(train); self._create_frequency_encoding(test)
        self.new_nums += [f'FREQ_{col}' for col in self.nums]
        self._create_arithmetic_interactions(train); self._create_arithmetic_interactions(test)
        self.new_nums += ['charges_deviation', 'monthly_to_total_ratio', 'avg_monthly_charges', 'cost_per_service', 'total_per_service']
        self._apply_distribution_features(train); self._apply_distribution_features(test)
        self.new_nums += self.dist_features
        self._apply_quantile_distance_features(train); self._apply_quantile_distance_features(test)
        self.new_nums += self.q_features
        self._create_digit_features(train); self._create_digit_features(test)
        for df_ in [train, test]:
            for c in self.digit_cols: df_[c] = df_[c].astype('float32')
        self.new_nums += self.digit_cols
        self._create_num_as_cat(train); self._create_num_as_cat(test)
        self._create_ngram_features(train); self._create_ngram_features(test)
        self.all_cat_cols = self.num_as_cat + self.cats + self.ngram_cols
        all_features = self.nums + self.cats + self.new_nums + self.num_as_cat + self.ngram_cols
        return train, test, all_features

    def transform(self, df):
        if not self.is_fitted: raise ValueError("Preprocessor not fitted.")
        if self.target_col in df.columns and not pd.api.types.is_numeric_dtype(df[self.target_col]):
            df[self.target_col] = df[self.target_col].astype(str).str.strip().str.capitalize().map({'No': 0, 'Yes': 1}).fillna(0).astype(int)
        if 'TotalCharges' in df.columns:
            df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(self.total_charges_median)
        self._create_service_counts(df)
        self._create_frequency_encoding(df)
        self._create_arithmetic_interactions(df)
        self._apply_distribution_features(df)
        self._apply_quantile_distance_features(df)
        self._create_digit_features(df)
        for c in self.digit_cols: df[c] = df[c].astype('float32')
        self._create_num_as_cat(df)
        self._create_ngram_features(df)
        return df

In [ ]:
# 2. TRAINING CONFIG & LOGIC

class CFG:
    TARGET = 'Churn'
    N_FOLDS = 5
    REPEATS = 2
    OPTUNA_TRIALS = 60
    RANDOM_SEED = 42
    # In Colab, if you upload to the root, path is just 'telco_customer_churn.csv'
    DATA_PATH = "data/telco_customer_churn.csv" 

LGB_BASE = {
    'random_state': CFG.RANDOM_SEED,
    'objective': 'binary',
    'metric': 'auc',
    'verbose': -1,
    'n_jobs': -1,
}

XGB_PARAMS = {
    'n_estimators': 1000,
    'learning_rate': 0.02,
    'max_depth': 7,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'n_jobs': -1,
    'random_state': CFG.RANDOM_SEED,
    'eval_metric': 'auc',
    'tree_method': 'hist',
    'device': 'cuda',  # Optimized for T4 GPU
}

CAT_BASE = {
    'random_seed': CFG.RANDOM_SEED,
    'verbose': False,
    'early_stopping_rounds': 50,
    'eval_metric': 'AUC',
    'auto_class_weights': 'Balanced',
    'task_type': 'GPU',         # Optimized for T4 GPU
}

def train_model():
    print("Loading dataset...")
    # Check root and data/ folder
    if os.path.exists(CFG.DATA_PATH):
        df = pd.read_csv(CFG.DATA_PATH)
    elif os.path.exists("data/" + CFG.DATA_PATH):
        df = pd.read_csv("data/" + CFG.DATA_PATH)
    else:
        print("Error: Dataset not found. Please upload 'telco_customer_churn.csv'.")
        from google.colab import files
        uploaded = files.upload()
        df = pd.read_csv(list(uploaded.keys())[0])
    
    if CFG.TARGET in df.columns and not pd.api.types.is_numeric_dtype(df[CFG.TARGET]):
        df[CFG.TARGET] = df[CFG.TARGET].astype(str).str.strip().str.capitalize().map({'No': 0, 'Yes': 1}).fillna(0).astype(int)

    churn_rate = df[CFG.TARGET].mean()
    pos_weight = (1 - churn_rate) / churn_rate
    print(f"Churn rate: {churn_rate:.3f} → scale_pos_weight = {pos_weight:.2f}")

    train, test = train_test_split(df, test_size=0.2, random_state=CFG.RANDOM_SEED, stratify=df[CFG.TARGET])
    train = train.reset_index(drop=True)
    test = test.reset_index(drop=True)
    orig = df.copy()

    preprocessor = Preprocessor(target_col=CFG.TARGET)
    train, test, features = preprocessor.fit_transform(train, test, orig)

    print("\n" + "="*70)
    print("OPTUNA HYPERPARAMETER TUNING + STACKING META-MODEL")
    print("="*70)

    X_full = train[features].copy()
    y_full = train[CFG.TARGET].values
    cat_cols = preprocessor.all_cat_cols

    def objective_lgb(trial):
        params = {
            **LGB_BASE,
            'n_estimators': trial.suggest_int('n_estimators', 800, 1500),
            'learning_rate': trial.suggest_float('learning_rate', 0.008, 0.08, log=True),
            'max_depth': trial.suggest_int('max_depth', 4, 9),
            'num_leaves': trial.suggest_int('num_leaves', 31, 128),
            'reg_alpha': trial.suggest_float('reg_alpha', 0.5, 5.0),
            'reg_lambda': trial.suggest_float('reg_lambda', 0.5, 5.0),
            'min_child_samples': trial.suggest_int('min_child_samples', 10, 40),
            'subsample': trial.suggest_float('subsample', 0.7, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 1.0),
            'scale_pos_weight': pos_weight,
        }
        inner_skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=CFG.RANDOM_SEED + trial.number)
        aucs = []
        for tr_idx, va_idx in inner_skf.split(X_full, y_full):
            X_tr_i = X_full.iloc[tr_idx].copy()
            X_va_i = X_full.iloc[va_idx].copy()
            y_tr_i = y_full[tr_idx]
            y_va_i = y_full[va_idx]
            te_i = TargetEncoder(cv=3, random_state=CFG.RANDOM_SEED)
            X_tr_i[cat_cols] = te_i.fit_transform(X_tr_i[cat_cols], y_tr_i)
            X_va_i[cat_cols] = te_i.transform(X_va_i[cat_cols])
            X_tr_num = X_tr_i.select_dtypes(include=[np.number])
            X_va_num = X_va_i.select_dtypes(include=[np.number])
            model = LGBMClassifier(**params)
            model.fit(X_tr_num, y_tr_i, eval_set=[(X_va_num, y_va_i)], callbacks=[early_stopping(50, verbose=False)])
            pred = model.predict_proba(X_va_num)[:, 1]
            aucs.append(roc_auc_score(y_va_i, pred))
        return np.mean(aucs)

    print("Tuning LightGBM...")
    study_lgb = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=CFG.RANDOM_SEED))
    study_lgb.optimize(objective_lgb, n_trials=CFG.OPTUNA_TRIALS)
    best_lgb_params = study_lgb.best_params
    print(f"   LGBM best AUC: {study_lgb.best_value:.5f}")

    def objective_cat(trial):
        params = {
            **CAT_BASE,
            'iterations': trial.suggest_int('iterations', 800, 1500),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
            'depth': trial.suggest_int('depth', 4, 8),
            'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 10),
        }
        inner_skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=CFG.RANDOM_SEED + trial.number)
        aucs = []
        for tr_idx, va_idx in inner_skf.split(X_full, y_full):
            X_tr_i = X_full.iloc[tr_idx].copy()
            X_va_i = X_full.iloc[va_idx].copy()
            y_tr_i = y_full[tr_idx]
            y_va_i = y_full[va_idx]
            cat_features = [col for col in cat_cols if col in X_tr_i.columns]
            for df_ in (X_tr_i, X_va_i):
                for c in cat_features: df_[c] = df_[c].astype(str)
            model = CatBoostClassifier(**params)
            model.fit(X_tr_i, y_tr_i, eval_set=(X_va_i, y_va_i), cat_features=cat_features, verbose=False)
            pred = model.predict_proba(X_va_i)[:, 1]
            aucs.append(roc_auc_score(y_va_i, pred))
        return np.mean(aucs)

    print("Tuning CatBoost...")
    study_cat = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=CFG.RANDOM_SEED))
    study_cat.optimize(objective_cat, n_trials=CFG.OPTUNA_TRIALS)
    best_cat_params = study_cat.best_params
    print(f"   CatBoost best AUC: {study_cat.best_value:.5f}")

    LGB_PARAMS = {**LGB_BASE, **best_lgb_params, 'scale_pos_weight': pos_weight}
    CAT_PARAMS = {**CAT_BASE, **best_cat_params}

    print("\n" + "="*70)
    print(f"TRAINING FINAL STACKED ENSEMBLE ({CFG.N_FOLDS}×{CFG.REPEATS})")
    print("="*70)

    skf_outer = RepeatedStratifiedKFold(n_splits=CFG.N_FOLDS, n_repeats=CFG.REPEATS, random_state=CFG.RANDOM_SEED)
    n_outer = CFG.N_FOLDS * CFG.REPEATS
    
    # For stacking
    oof_base_preds = np.zeros((len(train), 3)) # LGB, XGB, CAT
    test_base_preds = np.zeros((len(test), 3))
    
    lgb_models, xgb_models, cat_models = [], [], []
    t0 = time.time()

    for i, (train_idx, val_idx) in enumerate(skf_outer.split(train, train[CFG.TARGET])):
        print(f"\nOuter fold {i+1}/{n_outer}")
        X_tr = train.iloc[train_idx][features].copy()
        y_tr = train.iloc[train_idx][CFG.TARGET].values
        X_val = train.iloc[val_idx][features].copy()
        y_val = train.iloc[val_idx][CFG.TARGET].values
        X_te = test[features].copy()
        cat_cols = preprocessor.all_cat_cols
        
        # Target Encoding
        te = TargetEncoder(cv=3, random_state=CFG.RANDOM_SEED)
        X_tr_te = X_tr.copy(); X_val_te = X_val.copy(); X_te_te = X_te.copy()
        X_tr_te[cat_cols] = te.fit_transform(X_tr_te[cat_cols], y_tr)
        X_val_te[cat_cols] = te.transform(X_val_te[cat_cols])
        X_te_te[cat_cols] = te.transform(X_te_te[cat_cols])
        X_tr_num = X_tr_te.select_dtypes(include=[np.number])
        X_val_num = X_val_te.select_dtypes(include=[np.number])
        X_te_num = X_te_te.select_dtypes(include=[np.number])

        # LGBM
        lgb_model = LGBMClassifier(**LGB_PARAMS)
        lgb_model.fit(X_tr_num, y_tr, eval_set=[(X_val_num, y_val)], callbacks=[early_stopping(50, verbose=False)])
        lgb_val_p = lgb_model.predict_proba(X_val_num)[:, 1]
        lgb_test_p = lgb_model.predict_proba(X_te_num)[:, 1]
        lgb_models.append(lgb_model)

        # XGB
        xgb_model = XGBClassifier(**XGB_PARAMS, scale_pos_weight=pos_weight)
        xgb_model.fit(X_tr_num, y_tr, eval_set=[(X_val_num, y_val)], verbose=False)
        xgb_val_p = xgb_model.predict_proba(X_val_num)[:, 1]
        xgb_test_p = xgb_model.predict_proba(X_te_num)[:, 1]
        xgb_models.append(xgb_model)

        # CatBoost
        X_tr_cat = X_tr.copy(); X_val_cat = X_val.copy(); X_te_cat = X_te.copy()
        cat_features = [col for col in cat_cols if col in X_tr_cat.columns]
        for dfc in (X_tr_cat, X_val_cat, X_te_cat):
            for c in cat_features: dfc[c] = dfc[c].astype(str)
        cat_model = CatBoostClassifier(**CAT_PARAMS)
        cat_model.fit(X_tr_cat, y_tr, eval_set=(X_val_cat, y_val), cat_features=cat_features, verbose=False)
        cat_val_p = cat_model.predict_proba(X_val_cat)[:, 1]
        cat_test_p = cat_model.predict_proba(X_te_cat)[:, 1]
        cat_models.append(cat_model)

        # Store OOF predictions for Meta-model
        oof_base_preds[val_idx, 0] += lgb_val_p / CFG.REPEATS
        oof_base_preds[val_idx, 1] += xgb_val_p / CFG.REPEATS
        oof_base_preds[val_idx, 2] += cat_val_p / CFG.REPEATS
        
        test_base_preds[:, 0] += lgb_test_p / n_outer
        test_base_preds[:, 1] += xgb_test_p / n_outer
        test_base_preds[:, 2] += cat_test_p / n_outer
        
        fold_auc = roc_auc_score(y_val, np.mean([lgb_val_p, xgb_val_p, cat_val_p], axis=0))
        print(f"   Fold {i+1} Mean AUC: {fold_auc:.5f}")

    # Train Meta-model (Logistic Regression)
    print("\nTraining Meta-model (Logistic Regression)...")
    meta_model = LogisticRegression(random_state=CFG.RANDOM_SEED)
    meta_model.fit(oof_base_preds, train[CFG.TARGET])
    
    final_oof_preds = meta_model.predict_proba(oof_base_preds)[:, 1]
    final_test_preds = meta_model.predict_proba(test_base_preds)[:, 1]

    y_true = train[CFG.TARGET].values
    y_pred_binary = (final_oof_preds > 0.5).astype(int)
    
    te_final = TargetEncoder(random_state=CFG.RANDOM_SEED)
    te_final.fit(train[preprocessor.all_cat_cols], train[CFG.TARGET])

    bundle = {
        'lgb_models': lgb_models, 'xgb_models': xgb_models, 'cat_models': cat_models,
        'meta_model': meta_model,
        'preprocessor': preprocessor, 'te': te_final, 'features': features,
        'cols': X_tr_num.columns.tolist()
    }
    joblib.dump(bundle, 'churn_model.joblib')
    print("\n" + "="*70 + "\nTRAINING COMPLETE")
    print(f"Stacked OOF AUC: {roc_auc_score(y_true, final_oof_preds):.5f}")
    print(f"Accuracy:        {accuracy_score(y_true, y_pred_binary):.5f}\n" + "="*70)

if __name__ == "__main__":
    train_model()

Loading dataset...
Churn rate: 0.265 → scale_pos_weight = 2.77


[I 2026-03-26 13:23:42,862] A new study created in memory with name: no-name-33dc7f39-b52a-491d-9898-44514e3e228a



OPTUNA HYPERPARAMETER TUNING + STACKING META-MODEL
Tuning LightGBM...


[I 2026-03-26 13:23:44,705] Trial 0 finished with value: 0.8357544604549875 and parameters: {'n_estimators': 1062, 'learning_rate': 0.071417442434829, 'max_depth': 8, 'num_leaves': 89, 'reg_alpha': 1.2020838819909643, 'reg_lambda': 1.201975341512912, 'min_child_samples': 11, 'subsample': 0.9598528437324805, 'colsample_bytree': 0.8803345035229626}. Best is trial 0 with value: 0.8357544604549875.
[I 2026-03-26 13:23:48,341] Trial 1 finished with value: 0.8394584535523527 and parameters: {'n_estimators': 1296, 'learning_rate': 0.008388310180155688, 'max_depth': 9, 'num_leaves': 112, 'reg_alpha': 1.4555259980522428, 'reg_lambda': 1.318212352431953, 'min_child_samples': 15, 'subsample': 0.7912726728878613, 'colsample_bytree': 0.8574269294896714}. Best is trial 1 with value: 0.8394584535523527.
[I 2026-03-26 13:23:50,797] Trial 2 finished with value: 0.8399276905003137 and parameters: {'n_estimators': 1102, 'learning_rate': 0.01564296693019619, 'max_depth': 7, 'num_leaves': 44, 'reg_alpha': 

   LGBM best AUC: 0.84676
Tuning CatBoost...


...

   Fold 8 Mean AUC: 0.85246

Outer fold 9/10


Default metric period is 5 because AUC is/are not implemented for GPU


   Fold 9 Mean AUC: 0.83897

Outer fold 10/10


Default metric period is 5 because AUC is/are not implemented for GPU


   Fold 10 Mean AUC: 0.85030

Training Meta-model (Logistic Regression)...

TRAINING COMPLETE
Stacked OOF AUC: 0.84730
Accuracy:        0.80653


### Download the Model
Run the cell below to download the `churn_model.joblib` file to your computer.

In [ ]:
from google.colab import files
if os.path.exists('churn_model.joblib'):
    files.download('churn_model.joblib')
else:
    print("Model file not found. Did you run the training cell?")